# Nhóm 07 - INT1461 - Huấn luyện mô hình nhận diện lửa / khói
Notebook chạy trên Google Colab (Runtime > Change runtime type > GPU).
Quy trình: nạp tập ảnh từ Google Drive > chia tập theo nguồn video > huấn luyện MobileNetV2 > xuất model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Thư mục trên Drive chứa: data_raw.zip (data/raw/<lop>/...) và các tệp .py của thư mục ai/
THU_MUC = '/content/drive/MyDrive/Nhom07_INT1461'
%cd /content
!cp $THU_MUC/*.py /content/
!unzip -q -o $THU_MUC/data_raw.zip -d /content/

In [ ]:
!pip -q install paho-mqtt
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Chia tập theo nguồn video: 70% train, 15% val, 15% test
!python split_dataset.py --raw data/raw --out data/split --val 0.15 --test 0.15
!cat data/split/thong_ke.csv

In [ ]:
# Lưới ảnh mẫu của từng lớp (dùng cho hình minh họa trong báo cáo)
import os, random, matplotlib.pyplot as plt
from PIL import Image
LOP = ['binh_thuong', 'khoi', 'lua']
fig, ax = plt.subplots(3, 6, figsize=(12, 6))
for i, lop in enumerate(LOP):
    d = f'data/split/train/{lop}'
    for j, ten in enumerate(random.sample(os.listdir(d), 6)):
        ax[i, j].imshow(Image.open(os.path.join(d, ten))); ax[i, j].axis('off')
    ax[i, 0].set_title(lop, loc='left')
plt.tight_layout(); plt.savefig('luoi_anh_mau.png', dpi=120); plt.show()

In [ ]:
!python train_model.py --data data/split --out outputs --epochs1 10 --epochs2 10 --unfreeze 30

In [ ]:
from IPython.display import Image as Hien, display
display(Hien('outputs/duong_cong_huan_luyen.png'))
display(Hien('outputs/ma_tran_nham_lan.png'))
!cat outputs/metrics.json
!cp -r outputs luoi_anh_mau.png $THU_MUC/